In [1]:
import numpy as np; import pandas as pd; import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Conv2D, MaxPooling2D, Flatten, SimpleRNN, Reshape
from tensorflow.keras.callbacks import EarlyStopping, Callback
from tensorflow.keras.optimizers import Adam
import time
import warnings; warnings.filterwarnings('ignore')

In [2]:
class EpochTimer(Callback):
    def on_train_begin(self, logs = None): self.times = []
    def on_epoch_begin(self, epoch, logs = None): self._start = time.time()
    def on_epoch_end(self, epoch, logs = None): self.times.append(time.time() - self._start)

# Task 1
### Data Preparation and Preprocessing
- Load the fashion-MNIST dataset (training and testing) using the `load_data()` method from the `fmnist` package
- Convert the cloth items class labels into one-hot encoded format with 10 output classes
- Rescale inputs to the range $[0, 1]$
- Display the shape of the training and testing datasets

In [3]:
(x_train, y_train), (x_val, y_val) = fashion_mnist.load_data()
y_train_cat = to_categorical(y_train, 10); y_val_cat = to_categorical(y_val, 10)
x_train = x_train / 255; y_train = y_train / 255
print('Training shape:', x_train.shape); print('Test shape:', x_val.shape)

Training shape: (60000, 28, 28)
Test shape: (10000, 28, 28)


# Task 2
### FCFNN
- Reshape each image from 28 × 28 into a 1D vector of length 784
- Build an FCFNN using the `Sequential()` API from `keras` using a suitable combination of the following layers
  - `Dense()` layers with suitable number of  neurons
  - `Dropout()` layers with a suitable rates
  - `Dense()` output layer with 10 neurons and `'softmax'` activation
  - Begin with ReLU activation for all layers except the output layer
- Study the model architecture using the `summary()` method
- Compile the model using `Adam()` optimizer, `'categorical_crossentropy'` loss, `'accuracy'` as evaluation metric
- Train the model for suitable number of epochs with a suitable batch size, and provide the validation data separately during training
- Limit training of the model using `EarlyStopping()` from `keras` with a suitable tolerance
- Feel free to change the architecture of the model (number of layers, number of neurons, activation functions, batch size, number of epochs, early stopping specifics, optimizer learning rate, and so on) and try to improve the model
- Note down the final specifics of the model such as parameter count, average training time per epoch, and performance

In [4]:
x_train_fcfnn = x_train.reshape(-1, 784); x_val_fcfnn = x_val.reshape(-1, 784)
model_fcfnn = Sequential([Dense(units = 32, activation = 'relu', input_shape = (784,), name = 'dense_1'),
                          Dropout(rate = 0.05, name = 'dropout_1'),
                          Dense(units = 64, activation = 'relu', name = 'dense_2'),
                          Dropout(rate = 0.05, name = 'dropout_2'),
                          Dense(units = 10, activation = 'softmax', name = 'output')],
                         name = 'fcfnn')
model_fcfnn.summary()
fcfnn_params = model_fcfnn.count_params()

Model: "fcfnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 32)             │        25,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,882 (108.91 KB)

 Trainable params: 27,882 (108.91 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
early_stop = EarlyStopping(monitor = 'val_loss', patience = 3, restore_best_weights = True)

In [6]:
model_fcfnn.compile(optimizer = Adam(learning_rate = 0.001), loss = 'categorical_crossentropy', metrics = ['accuracy'])
timer = EpochTimer()
history_fcfnn = model_fcfnn.fit(x_train_fcfnn, y_train_cat, validation_split = 0.2,
                                epochs = 5, batch_size = 32, callbacks = [early_stop, timer])
avg_time_fcfnn = sum(timer.times) / len(timer.times)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.7862 - loss: 0.6032 - val_accuracy: 0.8307 - val_loss: 0.4522
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8454 - loss: 0.4260 - val_accuracy: 0.8621 - val_loss: 0.3829
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8587 - loss: 0.3877 - val_accuracy: 0.8663 - val_loss: 0.3714
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8646 - loss: 0.3669 - val_accuracy: 0.8696 - val_loss: 0.3594
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8706 - loss: 0.3525 - val_accuracy: 0.8668 - val_loss: 0.3715


In [7]:
fcfnn_train_loss, fcfnn_train_acc = model_fcfnn.evaluate(x_train_fcfnn, y_train_cat)
fcfnn_val_loss, fcfnn_val_acc = model_fcfnn.evaluate(x_val_fcfnn, y_val_cat)

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8805 - loss: 0.3264
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8444 - loss: 59.3932


In [8]:
hist_df = pd.DataFrame(history_fcfnn.history); hist_df.insert(0, 'epoch', np.arange(1, len(hist_df) + 1))
hist_df.set_index('epoch', inplace = True); hist_df

,accuracy,loss,val_accuracy,val_loss
epoch,,,,
1,0.786208,0.603173,0.830667,0.452229
2,0.845438,0.425961,0.862083,0.382916
3,0.858729,0.387711,0.866333,0.371440
4,0.864583,0.366894,0.869583,0.359436
5,0.870646,0.352535,0.866750,0.371481


# Task 3
### CNN
- Reshape the input images to 4D tensors with shape `(num_samples, 28, 28, 1)` to include the channel dimension
- Build a CNN using the `Sequential()` API from `keras` using a suitable combination of the following layers
  - `Conv2D()` layers with suitable number of units of suitable size and deafult stride
  - `MaxPooling2D()` layers with suitable size and default stride
  - `Flatten()` layer for feeding feature maps into additional `Dense()` and output layers
  - `Dense()` layers with suitable number of neurons
  - `Dropout()` layers with suitable rates
  - `Dense()` output layer with 10 neurons and `'softmax'` activation
  - Begin with ReLU activation for all layers except the output layer
- Study the model architecture using the `summary()` method
- Compile the model using `Adam()` optimizer, `'categorical_crossentropy'` loss, `'accuracy'` as evaluation metric
- Train the model for suitable number of epochs with a suitable batch size, and provide the validation data separately during training
- Limit training of the model using `EarlyStopping()` from `keras` with a suitable tolerance
- Feel free to change the architecture of the model (number of layers, number of units, activation functions, size and stride of kernels, batch size, number of epochs, early stopping specifics, optimizer learning rate, and so on) and try to improve the model
- Note down the final specifics of the model such as parameter count, average training time per epoch, and performance

In [9]:
x_train_cnn = x_train.reshape(-1, 28, 28, 1); x_val_cnn = x_val.reshape(-1, 28, 28, 1)
model_cnn = Sequential([Input(shape = (28, 28, 1), name = 'input'),
                        Conv2D(filters = 8, kernel_size = (3, 3), activation = 'relu', name = 'conv2d_1'),
                        MaxPooling2D(pool_size = (2, 2), name = 'maxpool2d_1'),
                        Conv2D(filters = 32, kernel_size = (3, 3), activation = 'relu', name = 'conv2d_2'),
                        MaxPooling2D(pool_size = (2, 2), name = 'maxpool2d_2'),
                        Flatten(name = 'flatten'),
                        Dense(units = 64, activation = 'relu', name = 'dense_1'),
                        Dropout(rate = 0.1, name = 'dropout_1'),
                        Dense(units = 10, activation = 'softmax', name = 'output')],
                       name = 'cnn')
model_cnn.summary()
cnn_params = model_cnn.count_params()

Model: "cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 26, 26, 8)      │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool2d_1 (MaxPooling2D)      │ (None, 13, 13, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 11, 11, 32)     │         2,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool2d_2 (MaxPooling2D)      │ (None, 5, 5, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 54,330 (212.23 KB)

 Trainable params: 54,330 (212.23 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model_cnn.compile(optimizer = Adam(learning_rate = 0.001), loss = 'categorical_crossentropy', metrics = ['accuracy'])
timer = EpochTimer()
history_cnn = model_cnn.fit(x_train_cnn, y_train_cat, validation_split = 0.2,
                            epochs = 5, batch_size = 32, callbacks = [early_stop, timer])
avg_time_cnn = sum(timer.times) / len(timer.times)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.7900 - loss: 0.5757 - val_accuracy: 0.8533 - val_loss: 0.4111
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.8554 - loss: 0.3966 - val_accuracy: 0.8727 - val_loss: 0.3440
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.8730 - loss: 0.3460 - val_accuracy: 0.8815 - val_loss: 0.3231
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.8842 - loss: 0.3175 - val_accuracy: 0.8862 - val_loss: 0.3063
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.8920 - loss: 0.2926 - val_accuracy: 0.8869 - val_loss: 0.3081


In [11]:
cnn_train_loss, cnn_train_acc = model_cnn.evaluate(x_train_cnn, y_train_cat)
cnn_val_loss, cnn_val_acc = model_cnn.evaluate(x_val_cnn, y_val_cat)

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8960 - loss: 0.2831
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8426 - loss: 42.9869


In [12]:
hist_df_cnn = pd.DataFrame(history_cnn.history); hist_df_cnn.insert(0, 'epoch', np.arange(1, len(hist_df_cnn) + 1))
hist_df_cnn.set_index('epoch', inplace = True); hist_df_cnn

,accuracy,loss,val_accuracy,val_loss
epoch,,,,
1,0.790000,0.575746,0.853333,0.411104
2,0.855396,0.396630,0.872750,0.344029
3,0.873000,0.345971,0.881500,0.323144
4,0.884229,0.317462,0.886167,0.306347
5,0.892042,0.292649,0.886917,0.308118


# Task 4
### RCNN
- Use the same input in this model as for CNN
- Build an RCNN using the `Sequential()` API from `keras` using a suitable combination of the following layers
  - `Conv2D()` layers with suitable number of units of suitable size and default stride
  - `MaxPooling2D()` layers with suitable size and default stride
  - `Reshape()` layer to transition from CNN feature map to RNN input
  - `SimpleRNN()` layers with suitable number of units
  - `Dense()` layers with suitable number of neurons
  - `Dropout()` layers with suitable rates
  - `Dense()` output layer with 10 neurons and `softmax` activation
  - Begin with ReLU activation for all layers except the output layer
- Study the model architecture using the `summary()` method
- Compile the model using `Adam()` optimizer, `'categorical_crossentropy'` loss, `'accuracy'` as evaluation metric
- Train the model for suitable number of epochs with a suitable batch size, and provide the validation data separately during training
- Limit training of the model using `EarlyStopping()` from `keras` with a suitable tolerance
- Feel free to change the architecture of the model (number of layers, number of units, size and stride of kernels, batch size, number of epochs, early stopping specifics, optimizer learning rate, and so on) and try to improve the model
- Note down the final specifics of the model such as parameter count, average training time per epoch, and performance

In [13]:
model_rcnn = Sequential([Input(shape = (28, 28, 1), name = 'input_cnn'),
                         Conv2D(filters = 8, kernel_size = (3, 3), activation = 'relu', name = 'conv2d_1'),
                         MaxPooling2D(pool_size = (2, 2), name = 'maxpool2d_1'),
                         Conv2D(filters = 32, kernel_size = (3, 3), activation = 'relu', name = 'conv2d_2'),
                         MaxPooling2D(pool_size = (2, 2), name = 'maxpool2d_2'),
                         Reshape(target_shape = (25, 32), name = 'input_rnn'),
                         SimpleRNN(units = 2, activation = 'relu', return_sequences = True, name = 'rnn_1'),
                         SimpleRNN(units = 4, activation = 'relu', return_sequences = False, name = 'rnn_2'),
                         Dense(units = 8, activation = 'relu', name = 'dense_1'),
                         Dropout(rate = 0.1, name = 'dropout_1'),
                         Dense(units = 10, activation = 'softmax', name = 'output')],
                        name = 'rcnn')
model_rcnn.summary()
rcnn_params = model_rcnn.count_params()

Model: "rcnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 26, 26, 8)      │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool2d_1 (MaxPooling2D)      │ (None, 13, 13, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 11, 11, 32)     │         2,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool2d_2 (MaxPooling2D)      │ (None, 5, 5, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ input_rnn (Reshape)             │ (None, 25, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_1 (SimpleRNN)               │ (None, 25, 2)          │            70 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_2 (SimpleRNN)               │ (None, 4)              │            28 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │            90 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,644 (10.33 KB)

 Trainable params: 2,644 (10.33 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model_rcnn.compile(optimizer = Adam(learning_rate = 0.001), loss = 'categorical_crossentropy', metrics = ['accuracy'])
timer = EpochTimer()
history_rcnn = model_rcnn.fit(x_train_cnn, y_train_cat, validation_split = 0.2,
                              epochs = 5, batch_size = 32, callbacks = [early_stop, timer])
avg_time_rcnn = sum(timer.times) / len(timer.times)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - accuracy: 0.2744 - loss: 1.7874 - val_accuracy: 0.3577 - val_loss: 1.5681
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - accuracy: 0.4630 - loss: 1.3541 - val_accuracy: 0.5832 - val_loss: 1.1633
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - accuracy: 0.5699 - loss: 1.1236 - val_accuracy: 0.6316 - val_loss: 0.9747


In [15]:
rcnn_train_loss, rcnn_train_acc = model_rcnn.evaluate(x_train_cnn, y_train_cat)
rcnn_val_loss, rcnn_val_acc = model_rcnn.evaluate(x_val_cnn, y_val_cat)

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.3620 - loss: 1.5669
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.1111 - loss: 82.2295


In [16]:
hist_df_rcnn = pd.DataFrame(history_rcnn.history); hist_df_rcnn.insert(0, 'epoch', np.arange(1, len(hist_df_rcnn) + 1))
hist_df_rcnn.set_index('epoch', inplace = True); hist_df_rcnn

,accuracy,loss,val_accuracy,val_loss
epoch,,,,
1,0.274354,1.787370,0.357750,1.568113
2,0.463042,1.354125,0.583167,1.163299
3,0.569854,1.123581,0.631583,0.974701


# Task 5
### Model Comparison
- Compare the three final models that were trained in terms of parameter count, average training time, loss and accuracy for training and validation sets
- Create a simple data frame for readability

In [17]:
df = pd.DataFrame(data = {'params': [fcfnn_params,  cnn_params,  rcnn_params],
                          'avg_time_s': [avg_time_fcfnn, avg_time_cnn, avg_time_rcnn],
                          'train_loss': [fcfnn_train_loss, cnn_train_loss, rcnn_train_loss],
                          'train_acc': [fcfnn_train_acc, cnn_train_acc, rcnn_train_acc],
                          'val_loss': [fcfnn_val_loss, cnn_val_loss, rcnn_val_loss],
                          'val_acc': [fcfnn_val_acc, cnn_val_acc, rcnn_val_acc]},
                  index = ['fcfnn', 'cnn', 'rcnn'])
df

,params,avg_time_s,train_loss,train_acc,val_loss,val_acc
fcfnn,27882,4.201530,0.326430,0.880483,59.393162,0.8444
cnn,54330,9.163336,0.283142,0.896000,42.986900,0.8426
rcnn,2644,19.206269,1.566934,0.361983,82.229492,0.1111
